# Mamba-2 (SSD) selective temporal SSM — Kaggle GPU fold

**Track E addendum to `cross-border-fraud`. GPU only — does not run on the CPU repo pipeline.**

The in-repo CPU study (`RESULTS.md` §10.1 / `scripts/06_ssm_selective.py`) found the
*selective* temporal SSM (`SelectiveTemporalSSM`, **0.780**) scores *below* the fixed
diagonal `TemporalSSM` (**0.806**), well under the `card_hour_rarity` oracle (**0.877**).
The conclusion — selectivity has nothing to exploit on a *stationary* per-card hour
distribution — was hedged on one optimisation caveat:

> *"TBPTT-64 starves the long-memory channel."*

This notebook **falsifies or confirms that hedge** with the canonical Mamba-2 / SSD
kernel under **full-sequence BPTT** (no truncation) and a **larger state bank**
(`d_state=128`) — the two things the local GTX-1650 / CPU box cannot do.

**Reading the result (either outcome sharpens a stated claim):**
- AUC **still ≤ ~0.806** → the inductive-bias claim *strengthens* (the TBPTT excuse is removed).
- AUC **closes toward 0.877** → §10.1 over-attributed the gap to inductive bias vs. an
  optimisation artifact.

Injection is seeded/deterministic, so the planted answer key is identical to the repo's.

### 1 · Environment (Kaggle: enable a GPU accelerator, internet ON)

In [ ]:
!pip -q install mamba-ssm causal-conv1d
import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle settings."
print(torch.__version__, torch.cuda.get_device_name(0))

### 2 · Repo + data (attach the `kartik2112/fraud-detection` dataset)

In [ ]:
import os, pathlib
!git clone -q https://github.com/vybhav72954/cross-border-fraud.git
os.chdir("cross-border-fraud")
# point the repo's expected data/raw/ at the attached Kaggle dataset
pathlib.Path("data/raw").mkdir(parents=True, exist_ok=True)
SRC = "/kaggle/input/fraud-detection"
for f in ("fraudTrain.csv", "fraudTest.csv"):
    dst = pathlib.Path("data/raw")/f
    if not dst.exists():
        os.symlink(f"{SRC}/{f}", dst)
print(sorted(os.listdir("data/raw")))

### 3 · Build the controlled dataset (deterministic injection → same answer key)

In [ ]:
!python scripts/00_build_dataset.py
import pandas as pd
tr = pd.read_parquet("data/processed/injected_train.parquet")
te = pd.read_parquet("data/processed/injected_test.parquet")
for d in (tr, te): d["trans_dt"] = pd.to_datetime(d["trans_date_trans_time"])
print("train", tr.shape, "test", te.shape)

### 4 · Per-card temporal sequences

Reuse the repo's `token_features` / `pack` so the input stream is *identical* to the
CPU selective SSM — `[one-hot hour | z(log dt) | z(log amt)]` per transaction, one
sequence per card. The only differences are the kernel (Mamba-2 SSD), **full-sequence
BPTT** (`max_seq=None`), and a larger `d_state`.

In [ ]:
import sys; sys.path.insert(0, ".")
import numpy as np
from src.models.sequence import token_features, pack
from src.inject import typology_dummies, TYPOLOGY_COL

y_tr = typology_dummies(tr)["temporal"].to_numpy()
feats_tr, scaler, grp_tr = token_features(tr, "temporal")
feats_te, _, grp_te = token_features(te, "temporal", scaler)
Xtr, mtr, _, ytr = pack(feats_tr, grp_tr, label=y_tr, max_seq=None)   # FULL sequences
Xte, mte, ridx_te = pack(feats_te, grp_te, label=None, max_seq=None)
print("cards", Xtr.shape[0], "max_len", Xtr.shape[1], "d_in", Xtr.shape[2])

### 5 · Mamba-2 model (full BPTT, `d_state=128`)

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from mamba_ssm import Mamba2

class Mamba2Temporal(nn.Module):
    def __init__(self, d_in, d_model=64, d_state=128, n_layers=2):
        super().__init__()
        self.inp = nn.Linear(d_in, d_model)
        self.norms = nn.ModuleList(nn.LayerNorm(d_model) for _ in range(n_layers))
        self.layers = nn.ModuleList(Mamba2(d_model=d_model, d_state=d_state)
                                    for _ in range(n_layers))
        self.head = nn.Linear(d_model, 1)
    def forward(self, x):                       # x (B, L, d_in); Mamba2 is causal
        h = self.inp(x)
        for norm, layer in zip(self.norms, self.layers):
            h = h + layer(norm(h))
        return self.head(h).squeeze(-1)

dev = "cuda"
model = Mamba2Temporal(Xtr.shape[2]).to(dev)
print(sum(p.numel() for p in model.parameters()), "params")

### 6 · Train (full-sequence BPTT) and score isolated solo-vs-legit AUC

In [ ]:
from sklearn.metrics import roc_auc_score

Xt, yt, mt = (torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(mtr))
npos = float(ytr[mtr].sum()); ntot = float(mtr.sum())
pos_w = torch.tensor([min((ntot - npos) / max(npos, 1.0), 50.0)]).to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
for ep in range(15):
    perm = np.random.permutation(len(Xt))
    for i in range(0, len(Xt), 48):
        sel = perm[i:i + 48]
        bx, by, bm = Xt[sel].to(dev), yt[sel].to(dev), mt[sel].to(dev)
        logit = model(bx)
        loss = F.binary_cross_entropy_with_logits(logit[bm], by[bm], pos_weight=pos_w)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {ep+1:2d}  loss {loss.item():.4f}")

# score the test split, scatter per-token probs back to rows
model.eval()
scores = np.zeros(len(te), dtype=np.float32)
Xte_t, mte_t = torch.tensor(Xte), torch.tensor(mte)
with torch.no_grad():
    for i in range(0, len(Xte_t), 48):
        sl = slice(i, i + 48)
        p = torch.sigmoid(model(Xte_t[sl].to(dev))).cpu().numpy()
        ri, mm = ridx_te[sl], mte[sl]
        scores[ri[mm]] = p[mm]

typ = te[TYPOLOGY_COL].fillna("").to_numpy()
solo, legit = typ == "temporal", typ == ""
m = solo | legit
auc = roc_auc_score(solo[m].astype(int), scores[m])
print(f"
Mamba-2 (SSD, full BPTT, d_state=128) isolated temporal AUC = {auc:.3f}")
print("reference line:  fixed 0.806  |  selective(CPU, TBPTT-64) 0.780  |  oracle 0.877")

### 7 · Interpretation

| outcome | what it means |
|---|---|
| AUC ≲ 0.806 | The TBPTT-64 hedge is **refuted**: even with full BPTT + a 128-d state, the selective kernel does not beat the fixed bank → the §10.1 inductive-bias claim *strengthens* (selectivity has nothing to exploit on a stationary per-card hour distribution). |
| 0.806 ≲ AUC ≲ 0.877 | The hedge is **supported**: part of the CPU regression was an optimisation artifact (truncated memory), not pure inductive bias — §10.1 should be softened accordingly. |

Either way this is a *falsification test of a stated caveat*, not a leaderboard chase —
the value is that the conclusion gets sharper, not that "the real kernel" was run.